In [3]:
from langchain_mistralai import ChatMistralAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [2]:
a = {'lang':['1','2','3']}
", ".join(a['lang'])

'1, 2, 3'

In [9]:
api_key = 'hIBq3oF9S5hz3YmoxEDmxK9OmZW91BSx'        
model = "mistral-large-latest"

model = ChatMistralAI(model=model, temperature=0.2, max_retries=2, api_key=api_key)


desc = """
Компания занимается поддержкой и автоматизацией инфраструктуры, 
используя Linux, Windows, Docker, Kubernetes, GitLab, Gitea, 
Prometheus, Grafana, Zabbix, bash, python;
английский средний, знание сетевых технологий, понимание разработки ПО, 
контейнеризации, CI/CD, написания скриптов, систем мониторинга; 
кандидат будет поддерживать и автоматизировать инфраструктуру, обслуживать серверы, 
обеспечивать безопасность, вводить новые сервисы, 
диагностировать проблемы, разрабатывать CI/CD пайплайны, 
работать с багтрекингом, строить серверную архитектуру, 
настраивать стенды; компания предлагает постоянную работу, 
полный день, работу на территории работодателя, командировки, 
возможность профессионального и карьерного роста, 
зарплату по результатам собеседования.
"""



def classification_object(name_topic: str=None):

    prompt = ChatPromptTemplate.from_messages(
            [("system", """
              Ты агент который помогает выделить самое главное из описания вакансий, обращая внимание на:
              
              Описание: Чем компания занимается, какие технологии использует, что она предлагает и какие требования (ожидания) от кандидата?
              Знание языка: какие языки требуются? ()
              Образование: требуется или не требуеться ?        
              
              ВАЖНО: Если в вакансии какой либо информации не указано, то просто оставляй None.
              Избався от всей не нужной информации. Например, даты, название компаний и тд.              
                            
              Описание: {context}      
              Твой ответ должен быть в одну строчку (описание, знание языка и образование должны быть разделены $):
              """)])
    

    chain = prompt | model | StrOutputParser()
    result = chain.invoke({"context": desc})
    return result

result=classification_object()
result

In [15]:
result.split('$')

['Компания занимается поддержкой и автоматизацией инфраструктуры, используя Linux, Windows, Docker, Kubernetes, GitLab, Gitea, Prometheus, Grafana, Zabbix, bash, python; кандидат будет поддерживать и автоматизировать инфраструктуру, обслуживать серверы, обеспечивать безопасность, вводить новые сервисы, диагностировать проблемы, разрабатывать CI/CD пайплайны, работать с багтрекингом, строить серверную архитектуру, настраивать стенды; знание сетевых технологий, понимание разработки ПО, контейнеризации, CI/CD, написания скриптов, систем мониторинга',
 'английский средний',
 'None']

In [1]:
import re

def extract_experience(text):
    """
    Извлекает опыт работы в месяцах из текста.
    """
    years = re.search(r'(\d+)\s*лет', text)
    months = re.search(r'(\d+)\s*месяц', text)
    
    years = int(years.group(1)) if years else 0
    months = int(months.group(1)) if months else 0
    
    total_months = years * 12 + months
    return total_months

# Пример данных
vacancy = "Опыт работы: От 3 до 6 лет"
resume = "Опыт работы: Опыт работы 10 лет 1 месяц"

# Преобразование
vacancy_range = re.findall(r'(\d+)', vacancy)
vacancy_min_months = int(vacancy_range[0]) * 12
vacancy_max_months = int(vacancy_range[1]) * 12

resume_months = extract_experience(resume)

# Результат
print(f"Вакансия: {vacancy_min_months}-{vacancy_max_months} месяцев")
print(f"Резюме: {resume_months} месяцев")

Вакансия: 36-72 месяцев
Резюме: 121 месяцев


In [6]:
result

'Разработка и внедрение систем управления проектами, координация работы команды, анализ и оптимизация бизнес-процессов, взаимодействие с клиентами и партнерами, подготовка отчетности и презентаций, управление бюджетом и ресурсами, проведение тренингов и обучения сотрудников.'

# Регулярки

In [30]:
import os
import sys
import re

In [10]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(parent_dir)

In [100]:
os.path.abspath(os.path.join(os.getcwd()))

'/Users/richardgurtsiev/Desktop/projects/save/delete_2024/del/dl_skip/advisor/DataCrafter/notebooks'

In [97]:
from typing import Union

SYNONYMS = {
    'backend': ['backend', 'бекэнд'],
    'frontend': ['frontend', 'фронтенд'],
    'devops': ['devops', 'девопс']
}

def classification_object(position: str) -> Union[str, tuple]:
    position = position.lower().replace('-', ' ')

    for standard_topic, synonyms in SYNONYMS.items():
        if any(synonym in position for synonym in synonyms):
            return standard_topic

    return position

def read_document(path):

    with open(path,'r+') as file:
        content = file.read()
        

In [86]:
path = ''.join((sys.path[-1],'/dataset/CV_DevOps.txt'))


def forge(topic: str):
    titles = []
    clasters = dict()

    with open(path,'r+') as file:
        content = file.read()

    job_blocks = content.split('-' * 50)

    for job in job_blocks:
        position_match = re.search(r'Ищет работу на должность: (.*?)\n', job)
        
        if not position_match:
            continue
        
        title = position_match.group(1)        
        class_obj = classification_object(title=title, topic=topic)
        clasters[class_obj] = clasters.get(class_obj, []) + [job.strip()]
        
        titles.append(class_obj)

    titles = [title.lower().replace('-',' ') for title in titles]
    unique_titles = list(set(titles))

    return unique_titles, clasters

In [89]:
unique_titles, clasters = forge('devops')

In [10]:
cv_content = {'api': [1, 2, 3], 'token': [1, 2, 3]}
document = {'api': [4, 5, 6], 'token': [4, 5, 6]}


for key in document:
    if key in cv_content:
        cv_content[key].extend(document[key])
    else:
        cv_content[key] = document[key]

print(cv_content)

{'api': [1, 2, 3, 4, 5, 6], 'token': [1, 2, 3, 4, 5, 6]}


In [13]:
import re 

text = """ 
Имя вакансии: Senior system analyst
Опыт работы: От 3 до 6 лет
Описание: Компания занимается созданием продукта для снабжения магазинов и складов на основе микросервисной архитектуры; использует SQL, реляционные БД, back-office системы, front-end системы, keycloack. Требования: знание SQL, опыт с реляционными БД, понимание 3ей нормальной формы, оптимизация запросов, понимание back-office систем, алгоритмов, авторизации, аутентификации, высоконагруженных систем, паттернов взаимодействия, проектирования систем, front-end систем, различий между толстым и тонким клиентами. Кандидат будет собирать и анализировать требования, работать с логикой расчета заказов, создавать интеграции, готовить инструкции, проводить демо и обучения, разрабатывать спецификацию API. Компания предлагает гибкую систему премирования, расширенный социальный пакет, автономность работы, возможность профессионального роста, корпоративное обучение, гибридный формат работы, современный офис.
Ключевые навыки: None
Тип занятости: Полная занятость
График работы: Полный день
Местоположение: Москва
Профессиональные роли: Системный аналитик
"""

match=r'Имя вакансии: (.*?)\n'

position_match = re.search(match, text)

position_match.group(1)


'Senior system analyst'

In [1]:
'back end' in 'back end разработчик на php (битрикс)'

True

In [2]:
def filter_dict(d, limit=3):
    """
    Ограничивает количество элементов в списке значений словаря до limit.
    Удаляет ключи, если длина их значений меньше limit.

    :param d: Исходный словарь
    :param limit: Максимальное количество элементов, которое должно остаться в списке
    :return: Обновленный словарь
    """
    result = {}
    for key, value in d.items():
        if len(value) >= limit:
            result[key] = value[:limit]
    return result

data = {
    "devops": [1, 2, 3, 4, 5, 6],
    "frontend": [1, 2],
    "backend": [1, 2, 3, 4]
}

filtered_data = filter_dict(data)
print(filtered_data)


{'devops': [1, 2, 3], 'backend': [1, 2, 3]}


In [4]:
if []:
    print('Condition is True')

In [7]:
from agent.vllm_server.openai_client import OpenAIClient
from agent.vllm_server.utils import api_base, openai_key
from agent.utils import LlmModelType

client = OpenAIClient(model_type=LlmModelType.QWEN, api_base=api_base, api_key=openai_key)
response = client.invoke(
    """
            Ты бот который отвечает на вопросы LLM Security
            ""","""
                3. Какая проблема связана с недостатками архитектуры RBAC в RAG-системах?
*
1 балл
Фильтрация запросов замедляет производительность
Сложность настройки и поддержки ролей
Дублирование данных
Все перечисленные проблемы
                """)

print(response)

Сложность настройки и поддержки ролей

Объяснение: В контексте RAG-систем (Retrieval-Augmented Generation), недостатки архитектуры RBAC (Role-Based Access Control) могут заключаться в сложности настройки и поддержки ролей. Это связано с тем, что при увеличении сложности системы и количества пользователей, необходимо правильно определить и настроить роли, а также управлять ими, что может быть затруднительным. Однако, важно отметить, что другие проблемы также могут быть актуальны в зависимости от конкретной реализации системы. Например, в некоторых случаях может возникать проблема дублирования данных или проблема фильтрации запросов, но в данном случае наиболее значимой является сложность настройки и поддержки ролей.
